In [7]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, Dense,
                                     Dropout, TimeDistributed, Flatten,
                                     Bidirectional, Input)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
data_path = '/content/drive/MyDrive/UCI HAR Dataset/'

In [10]:
import os

print("Files inside folder:")
print(os.listdir(data_path))

Files inside folder:
['train.csv', 'test.csv']


In [11]:
import pandas as pd

train_df = pd.read_csv(data_path + 'train.csv')
test_df = pd.read_csv(data_path + 'test.csv')

print(" Data Loaded Successfully!")
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

 Data Loaded Successfully!
Train Shape: (7352, 563)
Test Shape: (2947, 563)


In [12]:
X_train = train_df.iloc[:, :-1]
y_train = train_df.iloc[:, -1]
X_test = test_df.iloc[:, :-1]
y_test = test_df.iloc[:, -1]

In [13]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

activity_names = label_encoder.classes_

y_train_cat = to_categorical(y_train_encoded)
y_test_cat = to_categorical(y_test_encoded)

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
def create_segments(X, n_steps=7):
    n_features = X.shape[1]
    step_length = n_features // n_steps
    X = X[:, :step_length * n_steps]  # ensure divisible

    return X.reshape((X.shape[0], n_steps, step_length, 1))

n_steps = 7

X_train_seg = create_segments(X_train_scaled, n_steps)
X_test_seg = create_segments(X_test_scaled, n_steps)

print("New Shape:", X_train_seg.shape)

New Shape: (7352, 7, 80, 1)


In [16]:
model = Sequential()

model.add(Input(shape=(X_train_seg.shape[1],
                       X_train_seg.shape[2],
                       X_train_seg.shape[3])))

model.add(TimeDistributed(Conv1D(64, kernel_size=3, activation='relu')))
model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
model.add(TimeDistributed(Dropout(0.3)))

model.add(TimeDistributed(Conv1D(128, kernel_size=3, activation='relu')))
model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
model.add(TimeDistributed(Dropout(0.3)))

model.add(TimeDistributed(Flatten()))

In [17]:
model.add(Bidirectional(LSTM(128)))
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dense(len(activity_names), activation='softmax'))

In [18]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 7, 78, 64)      │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 7, 39, 64)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 7, 39, 64)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 7, 37, 128)     │        24,704 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 7, 18, 128)     │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 7, 18, 128)     │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 7, 2304)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │     2,491,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,533,190 (9.66 MB)

 Trainable params: 2,533,190 (9.66 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
early_stopping = EarlyStopping(monitor='val_loss',
                               patience=8,
                               restore_best_weights=True)

history = model.fit(
    X_train_seg, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test_seg, y_test_cat, verbose=0)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

y_pred = model.predict(X_test_seg)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:")
print(classification_report(y_test_encoded, y_pred_classes, target_names=activity_names))

Epoch 1/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 64s 591ms/step - accuracy: 0.6351 - loss: 0.8132 - val_accuracy: 0.8423 - val_loss: 0.4149
Epoch 2/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 51s 553ms/step - accuracy: 0.8349 - loss: 0.3889 - val_accuracy: 0.8980 - val_loss: 0.2632
Epoch 3/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 53s 580ms/step - accuracy: 0.8981 - loss: 0.2547 - val_accuracy: 0.8899 - val_loss: 0.2535
Epoch 4/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 86s 629ms/step - accuracy: 0.9259 - loss: 0.1911 - val_accuracy: 0.8912 - val_loss: 0.2622
Epoch 5/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 54s 586ms/step - accuracy: 0.9410 - loss: 0.1496 - val_accuracy: 0.9014 - val_loss: 0.2628
Epoch 6/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 54s 587ms/step - accuracy: 0.9498 - loss: 0.1289 - val_accuracy: 0.9184 - val_loss: 0.2339
Epoch 7/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 80s 562ms/step - accuracy: 0.9606 - loss: 0.1057 - val_accuracy: 0.9211 - val_loss: 0.2246
Epoch 8/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 81s 557ms/step - accuracy: 0.9626 - loss: 0.0988 - val_accu